# 举例：OpenRouter平台gpt模型能力画像

In [7]:
import os
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
from rich import print as rprint

from chapte01_summary.test import DEEPSEEK_API_KEY, DEEPSEEK_BASE_URL

# 从.env文件中加载环境变量
load_dotenv(override=True)
model = ChatOpenRouter(
    # model="openai/gpt-4o-mini",
    model="deepseek/deepseek-v3.2",
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=6
)

rprint(model.profile)

{
    'max_input_tokens': 163840,
    'max_output_tokens': 65536,
    'text_inputs': True,
    'image_inputs': False,
    'audio_inputs': False,
    'video_inputs': False,
    'text_outputs': True,
    'image_outputs': False,
    'audio_outputs': False,
    'video_outputs': False,
    'reasoning_output': True,
    'tool_calling': True,
    'structured_output': True
}

# 6.3 模型初始化参数(完整版)
## 6.3.1 查看所有初始化参数

In [4]:
from langchain_deepseek import ChatDeepSeek
print(ChatDeepSeek.model_fields.keys())

dict_keys(['name', 'cache', 'verbose', 'callbacks', 'tags', 'metadata', 'custom_get_token_ids', 'rate_limiter', 'disable_streaming', 'output_version', 'profile', 'client', 'async_client', 'root_client', 'root_async_client', 'model_name', 'temperature', 'model_kwargs', 'openai_api_key', 'openai_api_base', 'openai_organization', 'openai_proxy', 'request_timeout', 'stream_usage', 'max_retries', 'presence_penalty', 'frequency_penalty', 'seed', 'logprobs', 'top_logprobs', 'logit_bias', 'streaming', 'n', 'top_p', 'max_tokens', 'reasoning_effort', 'reasoning', 'verbosity', 'tiktoken_model_name', 'default_headers', 'default_query', 'http_client', 'http_async_client', 'stop', 'extra_body', 'include_response_headers', 'disabled_params', 'context_management', 'include', 'service_tier', 'store', 'truncation', 'use_previous_response_id', 'use_responses_api', 'api_key', 'api_base'])


In [5]:
from langchain_openai import ChatOpenAI
print(ChatOpenAI.model_fields.keys())

dict_keys(['name', 'cache', 'verbose', 'callbacks', 'tags', 'metadata', 'custom_get_token_ids', 'rate_limiter', 'disable_streaming', 'output_version', 'profile', 'client', 'async_client', 'root_client', 'root_async_client', 'model_name', 'temperature', 'model_kwargs', 'openai_api_key', 'openai_api_base', 'openai_organization', 'openai_proxy', 'request_timeout', 'stream_usage', 'max_retries', 'presence_penalty', 'frequency_penalty', 'seed', 'logprobs', 'top_logprobs', 'logit_bias', 'streaming', 'n', 'top_p', 'max_tokens', 'reasoning_effort', 'reasoning', 'verbosity', 'tiktoken_model_name', 'default_headers', 'default_query', 'http_client', 'http_async_client', 'stop', 'extra_body', 'include_response_headers', 'disabled_params', 'context_management', 'include', 'service_tier', 'store', 'truncation', 'use_previous_response_id', 'use_responses_api'])


## 6.3.2 模型类的参数构成

## config

In [8]:
from langchain.chat_models import init_chat_model

# 1. 初始化模型
load_dotenv(override=True)
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    temperature=0.2,
    max_tokens=500,
    # 指定可调整参数
    configurable_fields=("model", "model_provider", "temperature","max_tokens"),
)
# 2. 准备 config 字典
config = {
    "run_name": "joke_generation", # 在LangSmith中这次运行会显示为
    "joke_generation"
    "tags": ["tag1", "tag2"], # 打上标签便于分类查找
    "metadata": {"user_id": "123"}, # 记录用户ID
    "configurable":{
        "model": "deepseek-v4-pro", # 配置模型参数
        "model_provider": "openai", # 配置模型提供商参数
        "temperature": 0.7, # 配置温度参数
        "max_tokens": 1000 # 配置最大令牌数
    }
}
# 3. 调用模型并传入config
response = model.invoke(
    "1 + 2 = ？",
    config=config
)
rprint(response)

AIMessage(
    content='1 + 2 = 3',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 34,
            'prompt_tokens': 11,
            'total_tokens': 45,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 26,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 11
        },
        'model_provider': 'openai',
        'model_name': 'deepseek-v4-pro',
        'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
        'id': 'd190ebfa-d9fe-4ff3-b319-5c4ff396d92f',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f34cb-6289-7c81-891b-e23456e1045a-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 11,
        'output_tokens': 34,
        'total_tokens': 45,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 26}
    }
)

## 对话历史优化

In [9]:
def keep_recent_messages(messages,max_pairs = 3):
    """
    保留最近的 N 轮对话
    max_pairs: 保留的对话轮数（每轮 = user + assistant）
    """
    # 分离 system 和对话
    system_msgs = [m for m in messages if m.get("role") == "system"]
    conversation_msgs = [m for m in messages if m.get("role") != "system"]

    # 只保留最近的
    recent_msgs = conversation_msgs[-(max_pairs * 2):]
    return system_msgs + recent_msgs

In [10]:
# 初始化
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]
# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})

# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})
print(f"原始消息数: {len(long_conversation)}")

# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")

# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})

# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 你的第一个问题是：“列表和元组有什么区别？用一句解释”。


## 多轮对话聊天机器人
## 基于模型初始化、流式响应以及消息列表的拼接来创建多轮聊天机器人。

In [14]:
# 1. 基础配置
MAX_PAIRS_HISTORY = 10
EXIT_WORD = "quit"
## 3. 初始化消息列表
messages = [
{
    "role":"system",
    "content":"你是小谷姐姐，尚硅谷教育的数字员工，也是一名耐心、友好的智能助手。我会用自然、清晰的方式回答用户问题。"
    }
]
## 4. 启动提示
print(f"✨ 请输入问题，输入 {EXIT_WORD} 结束对话\n")
# 5. 多轮对话主循环
# 轮次记录
i = 1
while True:
    print("\n", "=" * 10, f'-> 第 {i} 轮对话开始 <-', "=" * 10, "\n")
    user_input = input("🙋 请输入：")
    # 退出判断
    if user_input.lower() == EXIT_WORD:
        print("🌙 对话已结束，欢迎下次再来！")
        break

    messages.append({"role": "user", "content": user_input})

    reply_content = ""

    keep_recent_messages(messages, max_pairs = MAX_PAIRS_HISTORY)
    response = model.stream(messages)

    for chunk in response:
        if chunk.content:
            print(chunk.content,end="",flush=True)
            reply_content += chunk.content

    messages.append({"role": "assistant", "content": reply_content})
    i += 1



✨ 请输入问题，输入 quit 结束对话


 ========== -> 第 1 轮对话开始 <- ========== 

1+1等于2哦，这是最基础的加法运算。😊 如果有什么需要进一步解释的，随时问我！
 ========== -> 第 2 轮对话开始 <- ========== 

你的第一个问题是“1+1等于几”，我已经回答过啦！😄 需要再回顾一下吗？
 ========== -> 第 3 轮对话开始 <- ========== 

抱歉，我无法实时查询天气数据，建议打开天气预报App或网站查看最新信息哦！如果需要其他帮助，随时告诉我～😊
 ========== -> 第 4 轮对话开始 <- ========== 

你的第一个问题是“1+1等于几”，我回答的是等于2。😊 需要我再说一遍吗？
 ========== -> 第 5 轮对话开始 <- ========== 

请问你指的是“出去”吗？如果是想问几时出去，我暂时没有时间概念哦～或者有其他意思？可以再说清楚一些，我帮你解答！😊
 ========== -> 第 6 轮对话开始 <- ========== 

🌙 对话已结束，欢迎下次再来！
